# V10 · 原1h入口的严格匹配支持复核

## tl;dr
普通Python逐格检查成功：全部251母保留；完整三配对154母
（61.35%），独立图上界同为154。
重新分配可恢复0母；90%需要226母。
这些数字来自下面的保存账本复核，不是收益回测。owner盈利目标尚未完成。
Jupyter内核执行及完整nbformat schema validation未完成，具体缺口见末节。

## Context & Methods
固定V10审计：251个原始1h K1母事件，每母三个控制，90%需要226母。
原匹配键、风险转移、当前/前一小时cross排除及严格foldEnd−72h边界不变。
本笔记本不重建这些特征，也不调用策略、MILP或任何交易执行器。

### Key Assumptions
保存图是上游已核对的合法事前边；此处验证其哈希和内部一致性，
不以再次读取原始行情证明边的因果性。全月分配是离线支持审计，不是实时分配。
其他最优分配可能选择不同母事件；一份解未选某母，不证明该母永远无法匹配。
同一母的阶段计数是有固定顺序的描述，不是删除各规则的因果效应。

## Data
只读下列固定目录中的summary.json及三个保存的审计CSV；不扫描其他文件，
不读取价格、收益、MFE、K2成败或策略输出。summary的SHA固定在下一格，
CSV字节必须与summary.output_hashes一致。上游价格哈希只作为来源身份，不重算。
从仓库任意子目录打开；也可先设置NOTEBOOK_REPOSITORY_ROOT为仓库路径。

In [1]:
import csv, gzip, hashlib, io, json, math
from collections import Counter, defaultdict, deque
from datetime import datetime, timezone
from pathlib import Path

RESULTS_RELATIVE = 'experiments/active/exp-btcusdtp-1h-matching-support-preholdout-20260906-v10/results'
SUMMARY_SHA256 = '6279ce97ac051e168e632291218a697ffc7558db611bcf5e007f23b51bb55440'
EVIDENCE_FILES = ('mother_audit.csv.gz', 'eligible_edges.csv.gz', 'maximum_allocation.csv.gz')
COUNT_PER_MOTHER = 3
EXPECTED_MOTHERS = 251
REQUIRED_COMPLETE = 226
def require(condition, message):
    if not condition:
        raise ValueError(message)
hint = globals().get("NOTEBOOK_REPOSITORY_ROOT")
roots = [Path(hint)] if hint is not None else [Path.cwd(), *Path.cwd().parents]
repository_root = next((p.resolve() for p in roots if (p / RESULTS_RELATIVE / "summary.json").is_file()), None)
require(repository_root is not None, "Run from the repository or set NOTEBOOK_REPOSITORY_ROOT")
results_directory = (repository_root / RESULTS_RELATIVE).resolve()
require(results_directory.is_relative_to(repository_root), "Evidence directory escaped repository")
def evidence_path(name):
    require(name in ("summary.json", *EVIDENCE_FILES), "File is not in the evidence allowlist")
    path = (results_directory / name).resolve()
    require(path.parent == results_directory, "Evidence symlink escaped its directory")
    return path
print("Saved support evidence:", RESULTS_RELATIVE)

Saved support evidence: experiments/active/exp-btcusdtp-1h-matching-support-preholdout-20260906-v10/results


### 1. 固定来源并读取保存账本

In [2]:
summary_bytes = evidence_path("summary.json").read_bytes()
require(hashlib.sha256(summary_bytes).hexdigest() == SUMMARY_SHA256, "Pinned summary hash mismatch")
def reject_json_constant(value):
    raise ValueError("Nonfinite JSON number: " + value)
summary = json.loads(summary_bytes, parse_constant=reject_json_constant)
require(summary["experiment_id"] == "exp-btcusdtp-1h-matching-support-preholdout-20260906-v10", "Wrong experiment")
for flag in ("outcomes_read_or_computed", "profitability_test", "holdout_consumed", "training_eligible", "production_eligible"):
    require(summary[flag] is False, "Unexpected outcome/production flag: " + flag)
require(summary["historical_full_parity"] is True and summary["original_assignment_feasible"] is True, "Missing upstream parity claim")
tables = {}
for name in EVIDENCE_FILES:
    payload = evidence_path(name).read_bytes()
    require(hashlib.sha256(payload).hexdigest() == summary["output_hashes"][name], "CSV hash mismatch: " + name)
    reader = csv.DictReader(io.StringIO(gzip.decompress(payload).decode("utf-8")))
    require(reader.fieldnames is not None and len(set(reader.fieldnames)) == len(reader.fieldnames), "Duplicate/missing CSV headers")
    rows = list(reader)
    require(all(None not in r and all(v is not None for v in r.values()) for r in rows), "Malformed CSV row")
    tables[name] = rows
mothers = tables["mother_audit.csv.gz"]
edges = tables["eligible_edges.csv.gz"]
allocation = tables["maximum_allocation.csv.gz"]
print("SHA256 verified: summary plus", len(tables), "saved CSVs")
print("Rows:", {name: len(rows) for name, rows in tables.items()})

SHA256 verified: summary plus 3 saved CSVs
Rows: {'mother_audit.csv.gz': 251, 'eligible_edges.csv.gz': 1829, 'maximum_allocation.csv.gz': 462}


### 2. 保留全部母事件并核对图、时钟与分配

In [3]:
def integer(value, allow_missing=False):
    if allow_missing and value == "":
        return None
    number = float(value)
    require(math.isfinite(number) and number >= 0 and number.is_integer(), "Invalid nonnegative integer")
    return int(number)
def stamp(value):
    parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
    require(parsed.tzinfo is not None, "Timestamp must have timezone")
    return parsed.astimezone(timezone.utc)
mother_by_id = {r["event_id"]: r for r in mothers}
require(len(mothers) == len(mother_by_id) == summary["mothers"] == EXPECTED_MOTHERS, "Lost/duplicated mother IDs")
require(all(mother_by_id), "Empty mother ID")
fold_ends = {"2023H1": "2023-07-01", "2023H2": "2024-01-01", "2024H1": "2024-07-01", "2024H2": "2025-01-01"}
fold_starts = {"2023H1": "2023-01-01", "2023H2": "2023-07-01", "2024H1": "2024-01-01", "2024H2": "2024-07-01"}
from datetime import timedelta
def valid_fold_time(value, fold):
    require(fold in fold_ends, "Unknown fold")
    time = stamp(value)
    start = stamp(fold_starts[fold] + "T00:00:00+00:00")
    end = stamp(fold_ends[fold] + "T00:00:00+00:00") - timedelta(hours=72)
    require(start <= time < end and time.minute == time.second == time.microsecond == 0, "Decision outside exact hourly fold/embargo")
    return time
for row in mothers:
    valid_fold_time(row["decision_time"], row["fold"])
edge_pairs = set()
neighbours = {mother: set() for mother in mother_by_id}
candidate_owners = defaultdict(set)
candidate_folds = defaultdict(set)
for row in edges:
    mother, candidate = row["event_id"], row["candidate_id"]
    require(mother in mother_by_id and candidate, "Orphan/empty graph ID")
    require(row["fold"] == mother_by_id[mother]["fold"], "Edge fold mismatch")
    time = valid_fold_time(candidate, row["fold"])
    require(time.isoformat() == candidate and stamp(row["candidate_time"]) == time, "Candidate identity/time mismatch")
    mother_time = stamp(mother_by_id[mother]["decision_time"])
    require((time.year, time.month, time.hour // 6) == (mother_time.year, mother_time.month, mother_time.hour // 6), "Different month/time bucket")
    pair = (mother, candidate)
    require(pair not in edge_pairs, "Duplicate admissible edge")
    edge_pairs.add(pair)
    neighbours[mother].add(candidate)
    candidate_owners[candidate].add(mother)
    candidate_folds[candidate].add(row["fold"])
require(all(len(folds) == 1 for folds in candidate_folds.values()), "Cross-fold control reuse")
require(len(edges) == summary["matching_edges"], "Edge count mismatch")
selected_pairs = [(r["event_id"], r["candidate_id"]) for r in allocation]
require(len(selected_pairs) == len(set(selected_pairs)), "Duplicate allocation edge")
require(set(selected_pairs) <= edge_pairs, "Allocated forbidden edge")
require(len({candidate for _, candidate in selected_pairs}) == len(allocation), "Control timestamp reused")
selected_counts = Counter(mother for mother, _ in selected_pairs)
require(all(n == COUNT_PER_MOTHER for n in selected_counts.values()), "Partial control groups")
require(all(r["fold"] == mother_by_id[r["event_id"]]["fold"] for r in allocation), "Allocation fold mismatch")
matched_count = len(selected_counts)
require(matched_count == summary["maximum_matched"], "Allocation and summary disagree")
require(len(allocation) == COUNT_PER_MOTHER * matched_count, "Wrong group size")
print("All", len(mothers), "mother IDs retained; complete allocation:", matched_count, "mothers /", len(allocation), "unique controls")

All 251 mother IDs retained; complete allocation: 154 mothers / 462 unique controls


## Results
下列输出来自保存账本的独立复算。缺失支持保留为未知可用数，不能伪装成零供给。
阶段分解使用报告相同的全部17个阶段，从same_month到unused_before，按固定次序
取首次低于3的位置；只有原可用数≥3、被先前分配消耗至不足3，才归于unused_before。
它不能与原始供给不足混为一谈；不合并fold_embargo与其他支持缺失。

In [4]:
status_counts = Counter(r["match_status"] for r in mothers)
require(dict(status_counts) == summary["old_status_counts"], "Historical status counts mismatch")
decomposition = Counter()
checkpoints = [(stage + "_count", stage) for stage in ('same_month', 'same_utc6h', 'same_vol_bucket', 'same_5m_colour', 'same_hourly_colour', 'same_slope', 'fold_embargo', 'vol_support', 'atr_support', 'entry_open_support', 'entry_continuity_support', 'five_minute_support', 'hourly_support', 'cross_exclusion', 'actual_mother_exclusion', 'positive_synthetic_stop', 'unused_before')]
for row in mothers:
    require(row["match_status"] == row["reconstructed_status"], "Reconstructed status mismatch")
    before = integer(row["preallocation_available"], allow_missing=True)
    available = integer(row["available_before_greedy"], allow_missing=True)
    degree = len(neighbours[row["event_id"]])
    if row["mother_search_reached"] == "False":
        require(before is None and available is None and degree == 0, "Missing support misrepresented as zero/known supply")
        require(integer(row["selected_count"]) == integer(row["assigned_controls"]) == 0, "Unknown mother cannot have assigned controls")
        require(row["match_status"] != "matched", "Unknown mother cannot be matched")
        decomposition["mother_missing_support"] += 1
        continue
    require(row["mother_search_reached"] == "True", "Unknown search flag")
    require(before == degree and available is not None, "Preallocation count does not equal graph degree")
    require(integer(row["used_before_count"]) + available == before, "Greedy consumption arithmetic mismatch")
    values = [integer(row[column]) for column, _ in checkpoints]
    require(all(a >= b for a, b in zip(values, values[1:])), "Nonmonotone ordered stage counts")
    require(values[-2:] == [before, available], "Stage availability mismatch")
    expected_selected = COUNT_PER_MOTHER if available >= COUNT_PER_MOTHER else 0
    require(integer(row["selected_count"]) == integer(row["assigned_controls"]) == expected_selected, "Greedy partial/reserved controls")
    require((row["match_status"] == "matched") == bool(expected_selected), "Greedy status inconsistent with availability")
    reason = next((label for (_, label), value in zip(checkpoints, values) if value < COUNT_PER_MOTHER), "matched")
    decomposition[reason] += 1
require(sum(decomposition.values()) == EXPECTED_MOTHERS, "Decomposition lost mothers")
require(status_counts["matched"] == summary["greedy_matched"], "Greedy count mismatch")
require(summary["greedy_controls"] == COUNT_PER_MOTHER * status_counts["matched"], "Historical group count mismatch")
print("Historical statuses:", dict(sorted(status_counts.items())))
print("Ordered support decomposition:", dict(sorted(decomposition.items())))

Historical statuses: {'insufficient_exact_controls': 94, 'matched': 154, 'missing_causal_matching_support': 3}
Ordered support decomposition: {'cross_exclusion': 71, 'fold_embargo': 1, 'matched': 154, 'mother_missing_support': 3, 'same_slope': 19, 'unused_before': 3}


### 3. 不调用求解器，重算连通分量容量上界

In [5]:
visited = set()
components = []
for initial in sorted(mother_by_id):
    if initial in visited:
        continue
    pending, component_mothers, component_candidates = deque([initial]), set(), set()
    while pending:
        mother = pending.popleft()
        if mother in visited:
            continue
        visited.add(mother)
        component_mothers.add(mother)
        for candidate in neighbours[mother]:
            if candidate not in component_candidates:
                component_candidates.add(candidate)
                pending.extend(candidate_owners[candidate] - visited)
    upper = min(len(component_mothers), len(component_candidates) // COUNT_PER_MOTHER)
    components.append((len(component_mothers), len(component_candidates), upper))
require(visited == set(mother_by_id), "BFS omitted isolated mothers")
upper_bound = sum(upper for _, _, upper in components)
require(upper_bound == summary["capacity"]["connected_component_upper_bound"], "Saved component bound mismatch")
require(matched_count <= upper_bound, "Feasible solution exceeds graph upper bound")
require(matched_count == upper_bound, "This graph bound is not tight; independent maximum NOT certified")
require(summary["capacity"]["optimal"] is True and summary["capacity"]["matched_mothers"] == matched_count, "Upstream certificate mismatch")
require(summary["required_complete_mothers"] == REQUIRED_COMPLETE, "Coverage target changed")
require(math.isclose(summary["maximum_coverage"], matched_count / EXPECTED_MOTHERS, rel_tol=0, abs_tol=1e-12), "Coverage denominator changed")
require(summary["coverage_gate_attainable"] is (matched_count >= REQUIRED_COMPLETE), "Coverage gate contradiction")
require(summary["allocation_recoverable"] == matched_count - status_counts["matched"], "Recoverable allocation mismatch")
print("Connected components:", len(components), "; independently proved upper bound:", upper_bound)
print("Feasible allocation reaches bound:", matched_count, "; maximum coverage:", round(100 * matched_count / EXPECTED_MOTHERS, 4), "%")
print("90% requires", REQUIRED_COMPLETE, "; additional complete mothers available by reallocation:", matched_count - status_counts["matched"])
print("Profitability was NOT tested; owner profit goal remains unachieved.")

Connected components: 231 ; independently proved upper bound: 154
Feasible allocation reaches bound: 154 ; maximum coverage: 61.3546 %
90% requires 226 ; additional complete mothers available by reallocation: 0
Profitability was NOT tested; owner profit goal remains unachieved.


## Takeaways
只有上一格全部核验成功，才由“合法完整分配下界=连通分量上界”独立证明固定图最大容量。
这不是再次运行MILP。此证明只属于保存图，不扩张到新月份、放宽后的匹配规则或收益表现。
支持不足不能通过降低90%门、减少控制或把可支持子集冒充251母全体来解决。
即使支持充分也不等于赚钱；本笔记本没有计算任何收益，owner盈利目标仍未完成。

### Execution gap
生成器的`--check`仅在一个普通Python命名空间从上到下执行并捕获真实stdout/stderr，
**不是Jupyter kernel execution，也没有完成完整nbformat schema validation**。
已检查最小4.5结构、合法唯一cell IDs与全部code compile。缺少nbformat、nbclient、ipykernel，未安装依赖。
在另一个已配好这些依赖的环境，可另行运行：

```bash
python -m jupyter nbconvert --execute --to notebook --inplace PATH_TO_NOTEBOOK.ipynb
```

完整schema验证还需`nbformat.validate(nbformat.read(path, as_version=4))`。
这两步尚未执行；不要把本次普通Python检查转述为Jupyter或浏览器验证通过。